# People aggregation

Use identifier-confirmed cross-source pairs to evaluate simple name matching. These pairs measure recall; they do not provide confirmed negative examples for measuring precision.

In [ ]:
import luigi
import pandas as pd

from pkb.tasks.build.people.identifier_matches import PeopleIdentifierMatchesTask

## Build the identifier validation pairs

In [ ]:
task = PeopleIdentifierMatchesTask()
luigi.build([task], local_scheduler=True)

matches = pd.read_csv(task.output().path, keep_default_na=False)
matches.shape

In [ ]:
matches.groupby(
    ["left_source", "right_source", "identifier_type"]
).size().rename("matches").reset_index()

In [ ]:
pair_columns = [
    "left_source",
    "left_source_id",
    "right_source",
    "right_source_id",
]
pairs = matches.drop_duplicates(pair_columns).copy()
pairs.shape

## Evaluate simple name rules

In [ ]:
same_normalised_name = (
    pairs["left_name_normalised"].ne("")
    & pairs["left_name_normalised"].eq(pairs["right_name_normalised"])
)

same_surname_and_initials = (
    pairs["left_surname"].ne("")
    & pairs["left_initials"].ne("")
    & pairs["left_surname"].eq(pairs["right_surname"])
    & pairs["left_initials"].eq(pairs["right_initials"])
)

pairs["same_normalised_name"] = same_normalised_name
pairs["same_surname_and_initials"] = same_surname_and_initials
pairs["name_match"] = same_normalised_name | same_surname_and_initials

In [ ]:
performance = pd.DataFrame(
    {
        "rule": [
            "same normalised name",
            "same surname and initials",
            "either rule",
        ],
        "matched": [
            same_normalised_name.sum(),
            same_surname_and_initials.sum(),
            pairs["name_match"].sum(),
        ],
    }
)
performance["identifier_matches"] = len(pairs)
performance["recall"] = performance["matched"] / len(pairs)
performance

In [ ]:
pairs.groupby(["left_source", "right_source"]).agg(
    identifier_matches=("identifier_value", "size"),
    name_matches=("name_match", "sum"),
    recall=("name_match", "mean"),
).reset_index()

## Inspect identifier matches missed by the name rules

In [ ]:
misses = pairs.loc[
    ~pairs["name_match"],
    [
        "left_source",
        "left_source_id",
        "left_name",
        "right_source",
        "right_source_id",
        "right_name",
        "left_birth_date",
        "right_birth_date",
        "identifier_type",
        "identifier_value",
    ],
]
misses.head(50)